# NoUs on Google Colab

上から順に ▶ を押して実行してください。**毎回のセッション開始時は「準備」の 1〜7 をすべて実行**します。

- データ（SQLite / ChromaDB）は Colab 内で処理し、**`save()` で Google Drive に保存**します。
- Colab は時間が経つと切断されて中身が消えます。**大きな処理のあとは必ず `save()`**。

## 準備 1. GPU の確認（任意）
`メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU` にしておくと LLM が速くなります。

In [ ]:
!nvidia-smi || echo 'GPU なし（CPU でも動きますが遅いです）'

## 準備 2. Google Drive を接続
許可画面が出たら、自分のアカウントを選んで許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 準備 3. 秘密情報（APIキー）の読み込み
左の 🔑（シークレット）に以下を登録し、「ノートブックからのアクセス」をオンにしてください。

| 名前 | 値 | 必須 |
|---|---|---|
| `OPENALEX_API_KEY` | OpenAlex の API キー | 推奨 |
| `GH_TOKEN` | GitHub の Personal Access Token | リポジトリが非公開の場合のみ |

In [ ]:
import os
from google.colab import userdata

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

for name in ("OPENALEX_API_KEY", "GH_TOKEN"):
    value = _secret(name)
    if value:
        os.environ[name] = value
    print(f"{name:18s}: {'OK' if value else '未設定'}")

## 準備 4. コードを取得

In [ ]:
import os
BRANCH = "main"
token = os.environ.get("GH_TOKEN")
url = (f"https://{token}@github.com/kohga1220-stack/NoUs.git" if token
       else "https://github.com/kohga1220-stack/NoUs.git")

if os.path.exists("/content/NoUs/.git"):
    !git -C /content/NoUs fetch -q && git -C /content/NoUs checkout -q {BRANCH} && git -C /content/NoUs pull -q
else:
    !git clone -q -b {BRANCH} {url} /content/NoUs
%cd /content/NoUs
!git log --oneline -1

## 準備 5. ライブラリをインストール（2〜4分）

In [ ]:
!pip install -q -r requirements.txt pytest
!python -m pytest -q 2>&1 | tail -1

## 準備 6. Ollama（ローカル LLM）を起動（初回 3〜5分）

In [ ]:
import subprocess, time, shutil
if not shutil.which("ollama"):
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull gemma4:e2b
!ollama list

## 準備 7. データを Drive から読み込む ＋ `save()` を定義
- 初回：旧データ `マイドライブ/AI/projects/nous/data` をコピー（元データはそのまま残ります）
- 2回目以降：`マイドライブ/AI/projects/NoUs_data`（`save()` の保存先）から読み込み

In [ ]:
import os, shutil
OLD_DATA   = "/content/drive/MyDrive/AI/projects/nous/data"
DRIVE_DATA = "/content/drive/MyDrive/AI/projects/NoUs_data"
LOCAL_DATA = "/content/NoUs/data"

src = DRIVE_DATA if os.path.exists(DRIVE_DATA) else OLD_DATA
if os.path.exists(src):
    shutil.copytree(src, LOCAL_DATA, dirs_exist_ok=True)
    print("読み込み元:", src)
else:
    os.makedirs(LOCAL_DATA, exist_ok=True)
    print("Drive にデータが見つからないので、空の状態から始めます")
print(os.listdir(LOCAL_DATA))

def save():
    """Colab 内のデータを Drive に保存（上書き）"""
    shutil.copytree(LOCAL_DATA, DRIVE_DATA, dirs_exist_ok=True)
    print("保存しました →", DRIVE_DATA)

---
# 実行
各ステップのあとに `save()` が入っています。

## A. 知識ベースの確認（旧データの索引を更新）

In [ ]:
!python nous.py embed
save()

## B. 構造の抽象化（まず10件で試す）
`[physics] Phase transition → threshold, ...` のように出れば成功。

In [ ]:
!python nous.py abstract 10

問題なければ残りすべて（件数により数十分）

In [ ]:
!python nous.py abstract
save()

## C. OpenAlex から各分野の論文を収集（まず各分野5件）

In [ ]:
!python nous.py openalex 5
!python nous.py embed
!python nous.py abstract
save()

## D. 分野間の空白地帯（Void）
**この出力をチャットに貼ってください。**

In [ ]:
!python nous.py voids --refresh
save()

## E. 空白地帯を架橋する論文の収集 ＋ 分野トレンド

In [ ]:
!python nous.py fillvoids --openalex
!python nous.py embed
!python nous.py abstract
!python nous.py trends
save()

## F. 討論（1回数分）
テーマは D の結果から選ぶのがおすすめ。`TOPIC` を書き換えて実行。

In [ ]:
TOPIC = "phase transition"
!python nous.py debate "{TOPIC}"
save()

## G. LLM 審査員による評価

In [ ]:
!python nous.py evaluate
!python nous.py reliability
save()

## G2. 分野名の統一（1回だけ実行）
旧データの `physics` などを OpenAlex の分野名にそろえます。何度実行しても安全です。

In [ ]:
!python nous.py migrate-domains
save()

## G3. 文献チェック（OpenAlex）
各仮説に最も近い既存論文を探し、文献に照らした新規性（lit-novelty）を記録します。1件約1秒・$0.001。

In [ ]:
!python nous.py litcheck
save()

## G3b. lit-novelty の較正（1回）
有名な理論10個を同じ方法で測り、「既出」の目安を作ります。約$0.01。`scores` に ⚑ が付くようになります。

In [ ]:
!python nous.py calibrate
save()

## G4. 別モデルの審査員を追加（任意）
仮説を作ったモデルとは別のモデルで採点すると、より独立した評価になります。

In [ ]:
!ollama pull llama3.2:3b
!ollama pull qwen2.5:3b
!python nous.py evaluate --judges gemma4:e2b,llama3.2:3b,qwen2.5:3b
save()

## H. 自分で採点（10件程度）
入力欄が出たら 1〜5 を入力して Enter。`s` = この仮説を飛ばす、`q` = 終了。

In [ ]:
from nous.evaluation.runner import human_rate
human_rate("kohga", limit=10)
save()

## I. 結果の確認
`scores` と `reliability` の出力もチャットに貼ってください。

In [ ]:
!python nous.py scores
!python nous.py reliability

## （任意）知識グラフの HTML をダウンロード

In [ ]:
!python nous.py graph
from google.colab import files
files.download("data/nous_full_graph.html")